In [ ]:
"""
Simulación Monte Carlo del modelo de Ising 2D con red cudrada L x L con condiciones de frontera periódicas usando el algoritmo de Metropolis.

Hamiltoniano: H = -J * sum_{<i,j>} s_i s_j ,  s_i = +-1"""

import numpy as np


def random_config(L, rng):
    """Configuración inicial de espines."""
    return rng.choice(np.array([-1, 1], dtype=np.int8), size=(L, L))


def delta_E(espines, i, j, L):
    """Cambio de energía si se voltea el espín."""
    s = espines[i, j]
    vecinos= (
        espines[(i + 1) % L, j]
        + espines[(i - 1) % L, j]
        + espines[i, (j + 1) % L]
        + espines[i, (j - 1) % L]
    )
    # H = -J s * sum(vecinos); al voltear s -> -s, dE = 2 J s * sum(vecinos), J=1
    return 2.0 * s * vecinos





def total_energy(espines, L):
    """Energía total"""
    right = np.roll(espines, -1, axis=1)
    down = np.roll(espines, -1, axis=0)
    return -np.sum(espines * right) - np.sum(espines * down)


def metropolis_sweep(espines, L, beta, rng):
    """Monte Carlo con L*L intentos de volteo (Metropolis)."""
    for _ in range(L * L):
        i = rng.integers(0, L)
        j = rng.integers(0, L)
        dE = delta_E(espines, i, j, L)
        if dE <= 0 or rng.random() < np.exp(-beta * dE):
            espines[i, j] *= -1
    return espines


def run_simulation(L, T, n_therm, n_measure, measure_every=5, seed=None, espines0=None):
    """
    Corre la simulación a temperatura T.
    Devuelve: espines finales, arreglo de energía por espín (toda la corrida,
    para la gráfica de equilibración) y promedios <|E|>/N, <|m|> tras termalizar.
    """
    rng = np.random.default_rng(seed)
    beta = 1.0 / T
    N = L * L

    espines = espines0.copy() if espines0 is not None else random_config(L, rng)

    energy_trace = np.empty(n_therm + n_measure)

    # Termalizacion
    for t in range(n_therm):
        espines = metropolis_sweep(espines, L, beta, rng)
        energy_trace[t] = total_energy(espines, L) / N

    # Medicion
    e_samples, m_samples = [], []
    for t in range(n_measure):
        espines = metropolis_sweep(espines, L, beta, rng)
        e = total_energy(espines, L) / N
        m = np.sum(espines) / N
        energy_trace[n_therm + t] = e
        if t % measure_every == 0:
            e_samples.append(e)
            m_samples.append(m)

    e_samples = np.array(e_samples)
    m_samples = np.array(m_samples)

    return {
        "espines": espines,
        "energy_trace": energy_trace,
        "E_mean": e_samples.mean(),
        "E_std": e_samples.std(ddof=1) / np.sqrt(len(e_samples)),
        "m_abs_mean": np.abs(m_samples).mean(),
        "m_abs_std": np.abs(m_samples).std(ddof=1) / np.sqrt(len(m_samples)),
    }


if __name__ == "__main__":
    L = 32
    Tc = 2.0 / np.log(1 + np.sqrt(2))
    print(f"Tc teorica = {Tc:.4f}")

    result = run_simulation(L, T=2.5, n_therm=2000, n_measure=8000, seed=0)
    print("E/N =", result["E_mean"], "+-", result["E_std"])
    print("|m| =", result["m_abs_mean"], "+-", result["m_abs_std"])
